In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

df = pd.read_csv("../data/cs-training.csv", index_col=0)

In [ ]:
# Checking missingness vs. default rate first
df.groupby(df["MonthlyIncome"].isna())["SeriousDlqin2yrs"].mean()

# Drop the impossible rows:
df = df[df["age"] > 0] # Keeps every row where age is actually greater than 0

In [8]:
# Handle sentinel codes in late-payment columns

late_cols = [
    "NumberOfTime30-59DaysPastDueNotWorse",
    "NumberOfTime60-89DaysPastDueNotWorse",
    "NumberOfTimes90DaysLate",
]

for col in late_cols:
    df.loc[df[col].isin([96,98]), col] = np.nan

In [9]:
# Impute missing values

df["MonthlyIncome"] = df["MonthlyIncome"].fillna(df["MonthlyIncome"].median())
df["NumberOfDependents"] = df["NumberOfDependents"].fillna(0)
for col in late_cols:
    df[col] = df[col].fillna(df[col].median())

In [10]:
# Cap outliers:
for col in ["RevolvingUtilizationOfUnsecuredLines", "DebtRatio"]:
    cap = df[col].quantile(0.99)
    df[col] = df[col].clip(upper=cap)

In [11]:
# Split
X = df.drop(columns=["SeriousDlqin2yrs"])
y = df["SeriousDlqin2yrs"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Save outputs
train = X_train.copy()
train["SeriousDlqin2yrs"] = y_train
test = X_test.copy()
test["SeriousDlqin2yrs"] = y_test

train.to_csv("../data/train.csv")
test.to_csv("../data/test.csv")

In [13]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 149999 entries, 1 to 150000
Data columns (total 11 columns):
 #   Column                                Non-Null Count   Dtype  
---  ------                                --------------   -----  
 0   SeriousDlqin2yrs                      149999 non-null  int64  
 1   RevolvingUtilizationOfUnsecuredLines  149999 non-null  float64
 2   age                                   149999 non-null  int64  
 3   NumberOfTime30-59DaysPastDueNotWorse  149999 non-null  float64
 4   DebtRatio                             149999 non-null  float64
 5   MonthlyIncome                         149999 non-null  float64
 6   NumberOfOpenCreditLinesAndLoans       149999 non-null  int64  
 7   NumberOfTimes90DaysLate               149999 non-null  float64
 8   NumberRealEstateLoansOrLines          149999 non-null  int64  
 9   NumberOfTime60-89DaysPastDueNotWorse  149999 non-null  float64
 10  NumberOfDependents                    149999 non-null  float64
dtypes: fl